In [0]:
from pyspark.sql.functions import *
from delta.tables import DeltaTable


def build_gold(batch_df, batch_id):

    print(f"========== GOLD BATCH {batch_id} STARTED ==========")

    # Read the complete current Silver state
    silver_current = spark.table(
        "cdc_catalog.silver.silver_customer"
    )

    print(f"Silver count: {silver_current.count()}")

    # Build Gold aggregation
    gold_df = (
        silver_current
        .groupBy("city")
        .agg(
            count("*").alias("customer_count"),
            sum("salary").alias("total_salary"),
            avg("salary").alias("average_salary")
        )
    )

    print("Gold aggregation created")
    gold_df.show(truncate=False)

    gold_table = "cdc_catalog.gold.gold_customer"

    # Check whether Gold table exists
    table_exists = spark.catalog.tableExists(gold_table)

    print(f"Gold table exists: {table_exists}")

    if not table_exists:

        print("Gold table does not exist. Creating it...")

        (
            gold_df
            .write
            .format("delta")
            .mode("overwrite")
            .saveAsTable(gold_table)
        )

        print("Gold table created successfully")

    else:

        print("Gold table exists. Performing MERGE...")

        gold_target = DeltaTable.forName(
            spark,
            gold_table
        )

        (
            gold_target.alias("g")
            .merge(
                gold_df.alias("s"),
                "g.city = s.city"
            )
            .whenMatchedUpdateAll()
            .whenNotMatchedInsertAll()
            .whenNotMatchedBySourceDelete()
            .execute()
        )

        print("Gold MERGE completed successfully")

    print(f"========== GOLD BATCH {batch_id} COMPLETED ==========")

In [0]:
gold_stream = (
    spark.readStream
    .table( "cdc_catalog.silver.silver_customer")
)

gold_query = (
    gold_stream
    .writeStream
    .foreachBatch(build_gold)
    .option(
        "checkpointLocation",
        "/Volumes/cdc_catalog/cdc_demo/cdc_files/_checkpoint/gold"
        )
    .trigger(availableNow=True)
    .start()
)

gold_query.awaitTermination()